In [ ]:
'''File with code removed from data_comprovations.ipynb that right now is not needed but it might be useful later.'''

In [ ]:
import sys
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Set, Tuple

# Minimal import setup: ensure the notebook folder is importable.
_current = Path.cwd()
if (_current / "gtfs_utils.py").exists():
    _import_dir = _current
elif (_current / "data-comprovations" / "gtfs_utils.py").exists():
    _import_dir = _current / "data-comprovations"
else:
    raise FileNotFoundError("gtfs_utils.py not found.")

if str(_import_dir) not in sys.path:
    sys.path.insert(0, str(_import_dir))

from gtfs_utils import (
    ROUTES_FILE,
    STOPS_FILE,
    STOP_TIMES_CLEANED_FILE,
    TRIPS_CLEANED_FILE,
    check_missing_files,
    load_stop_names,
    read_dict_rows,
)

# Route terminals by route_id: (first_terminal_stop_id, last_terminal_stop_id)
ROUTE_TERMINAL_STOPS: Dict[str, Tuple[str, str]] = {
    "1.1.1": ("1.111", "1.140"),
    "1.2.1": ("1.210", "1.227"),
    "1.3.1": ("1.314", "1.339"),
    "1.4.1": ("1.413", "1.434"),
    "1.5.1": ("1.509", "1.534"),
    "1.91.1": ("1.901", "1.918"),
    "1.94.1": ("1.930", "1.945"),
    "1.101.1": ("1.951", "1.916"),
    "1.104.1": ("1.930", "1.936"),
    "1.11.1": ("1.1136", "1.1140"),
    "1.99.1": ("1.9901", "1.9902"),
}

#### Do all trips begin and end at the final stops?

Goal: check that each trip in trips_cleaned.txt (prefix '1.') starts and ends at the terminal
stops defined for its route_id in ROUTE_TERMINAL_STOPS.

Method: for each trip_id, we read stop_times_cleaned.txt, take the minimum and maximum
stop_sequence rows, and compare their stop_id values against the two expected route terminals.

Output: grouped mismatch summary by line (route_short_name) and observed start/end station
names. For each group, show departure times from the station that is not one of the two expected
terminals.

In [ ]:
def main():
    check_missing_files([TRIPS_CLEANED_FILE, STOP_TIMES_CLEANED_FILE, ROUTES_FILE, STOPS_FILE])

    # route_id -> route_short_name (only for route_id in the terminal dictionary)
    route_short_name: Dict[str, str] = {}
    for row in read_dict_rows(ROUTES_FILE):
        rid = row.get('route_id', '').strip()
        if rid in ROUTE_TERMINAL_STOPS:
            route_short_name[rid] = row.get('route_short_name', '').strip()

    # trip_id -> route_id from trips_cleaned.txt (only trip_id with prefix '1.' and route_id in scope)
    trip_to_route: Dict[str, str] = {}
    total_trips_rows = 0
    for row in read_dict_rows(TRIPS_CLEANED_FILE):
        total_trips_rows += 1
        trip_id = row.get('trip_id', '').strip()
        route_id = row.get('route_id', '').strip()
        if not trip_id.startswith('1.') or route_id not in ROUTE_TERMINAL_STOPS:
            continue
        trip_to_route[trip_id] = route_id

    if not trip_to_route:
        print("No trips found in trips_cleaned.txt for route_id keys in ROUTE_TERMINAL_STOPS.")
        return

    # stop_id -> stop_name for friendly reporting
    stop_names = load_stop_names(STOPS_FILE)

    routes_in_scope = set(trip_to_route.values())

    # Keep line order following ROUTE_TERMINAL_STOPS insertion order.
    line_preferred_terminal_order: Dict[str, List[str]] = {}
    line_order: List[str] = []
    seen_lines: Set[str] = set()
    for route_id, (exp_start, exp_end) in ROUTE_TERMINAL_STOPS.items():
        if route_id not in routes_in_scope:
            continue
        line_name = route_short_name.get(route_id, route_id)
        if line_name in seen_lines:
            continue
        seen_lines.add(line_name)
        line_order.append(line_name)
        line_preferred_terminal_order[line_name] = [
            stop_names.get(exp_start, "(no name)"),
            stop_names.get(exp_end, "(no name)"),
        ]

    # For each trip_id, keep only first and last stop_sequence rows
    # trip_id -> (min_seq, min_stop_id, min_dep, max_seq, max_stop_id, max_dep)
    trip_bounds: Dict[str, Tuple[int, str, str, int, str, str]] = {}
    scanned_stop_times_rows = 0
    for row in read_dict_rows(STOP_TIMES_CLEANED_FILE):
        scanned_stop_times_rows += 1
        trip_id = row.get('trip_id', '').strip()
        if trip_id not in trip_to_route:
            continue

        seq_raw = row.get('stop_sequence', '').strip()
        try:
            seq = int(seq_raw)
        except Exception:
            continue

        stop_id = row.get('stop_id', '').strip()
        dep_time = row.get('departure_time', '').strip()

        if trip_id not in trip_bounds:
            trip_bounds[trip_id] = (seq, stop_id, dep_time, seq, stop_id, dep_time)
            continue

        min_seq, min_sid, min_dep, max_seq, max_sid, max_dep = trip_bounds[trip_id]
        if seq < min_seq:
            min_seq, min_sid, min_dep = seq, stop_id, dep_time
        if seq > max_seq:
            max_seq, max_sid, max_dep = seq, stop_id, dep_time
        trip_bounds[trip_id] = (min_seq, min_sid, min_dep, max_seq, max_sid, max_dep)

    print(f"Rows in trips_cleaned.txt: {total_trips_rows}")
    print(f"Trips in scope (prefix '1.' and route with terminal dictionary): {len(trip_to_route)}")
    print(f"Rows scanned in stop_times_cleaned.txt: {scanned_stop_times_rows}")
    print(f"Trips with stop_times found: {len(trip_bounds)}")

    missing_in_stop_times = sorted(tid for tid in trip_to_route if tid not in trip_bounds)
    if missing_in_stop_times:
        print(f"WARNING: {len(missing_in_stop_times)} trips in scope have no rows in stop_times_cleaned.txt.")

    # Grouped mismatches: line -> (start_name, end_name) -> {count, non_terminal_times_by_station}
    grouped: Dict[str, Dict[Tuple[str, str], Dict[str, object]]] = defaultdict(dict)
    total_violations = 0

    for trip_id in sorted(trip_bounds):
        route_id = trip_to_route[trip_id]
        line_name = route_short_name.get(route_id, route_id)
        expected_start, expected_end = ROUTE_TERMINAL_STOPS[route_id]
        expected = {expected_start, expected_end}

        _, start_sid, start_dep, _, end_sid, end_dep = trip_bounds[trip_id]
        observed = {start_sid, end_sid}

        if observed == expected:
            continue

        total_violations += 1
        start_name = stop_names.get(start_sid, "(no name)")
        end_name = stop_names.get(end_sid, "(no name)")
        pattern_key = (start_name, end_name)

        line_groups = grouped.setdefault(line_name, {})
        if pattern_key not in line_groups:
            line_groups[pattern_key] = {
                "count": 0,
                "non_terminal_times": defaultdict(list),
            }

        item = line_groups[pattern_key]
        item["count"] = int(item["count"]) + 1
        non_terminal_times = item["non_terminal_times"]

        # Collect departure times from stations that are not expected terminals.
        if start_sid not in expected:
            non_terminal_times[start_name].append((start_dep or "")[:5])
        if end_sid not in expected:
            non_terminal_times[end_name].append((end_dep or "")[:5])

    if total_violations == 0:
        print("All correct: every checked trip starts and ends at the expected terminal stops for its route_id.")
        return

    print(f"MISSING terminal consistency in {total_violations} trips:")

    ordered_lines = [ln for ln in line_order if ln in grouped]
    remaining_lines = sorted(ln for ln in grouped.keys() if ln not in set(ordered_lines))
    final_line_order = ordered_lines + remaining_lines

    for line_name in final_line_order:
        line_groups = grouped[line_name]
        line_total = sum(int(v["count"]) for v in line_groups.values())
        print(f"\nLine {line_name}: {line_total} trips with terminal mismatch")

        preferred = line_preferred_terminal_order.get(line_name, [])
        preferred_pos = {name: i for i, name in enumerate(preferred)}

        def pattern_sort_key(item: Tuple[Tuple[str, str], Dict[str, object]]) -> Tuple[int, int, int, int, str, str]:
            (start_name, end_name), info = item
            count_key = -int(info["count"])

            start_in_terminal = start_name in preferred_pos
            end_in_terminal = end_name in preferred_pos

            # 0: grouped by ending terminal
            # 1: grouped by starting terminal
            # 2: neither side matches expected terminals
            if end_in_terminal and not start_in_terminal:
                return (0, preferred_pos[end_name], 0, count_key, start_name, end_name)
            if start_in_terminal and not end_in_terminal:
                return (1, preferred_pos[start_name], 0, count_key, end_name, start_name)
            return (2, 0, 0, count_key, start_name, end_name)

        sorted_patterns = sorted(line_groups.items(), key=pattern_sort_key)

        for (start_name, end_name), info in sorted_patterns:
            times_flat: List[str] = []
            for station_name in sorted(info["non_terminal_times"].keys()):
                times_flat.extend(t for t in info["non_terminal_times"][station_name] if t)

            unique_times = []
            seen = set()
            for t in sorted(times_flat):
                if t not in seen:
                    seen.add(t)
                    unique_times.append(t)

            preview_limit = 3
            shown = unique_times[:preview_limit]
            suffix = "..." if len(unique_times) > preview_limit else ""
            joined = ", ".join(shown) if shown else "(none)"
            print(f"- {start_name} - {end_name} ({joined}{suffix})")

main()

In [ ]:
# Case breakdown for the same terminal-check logic (all trips in scope).

def main():
    check_missing_files([TRIPS_CLEANED_FILE, STOP_TIMES_CLEANED_FILE, ROUTES_FILE, STOPS_FILE])

    route_short_name: Dict[str, str] = {}
    for row in read_dict_rows(ROUTES_FILE):
        rid = row.get('route_id', '').strip()
        if rid in ROUTE_TERMINAL_STOPS:
            route_short_name[rid] = row.get('route_short_name', '').strip()

    trip_to_route: Dict[str, str] = {}
    for row in read_dict_rows(TRIPS_CLEANED_FILE):
        trip_id = row.get('trip_id', '').strip()
        route_id = row.get('route_id', '').strip()
        if trip_id.startswith('1.') and route_id in ROUTE_TERMINAL_STOPS:
            trip_to_route[trip_id] = route_id

    if not trip_to_route:
        print("No trips found in trips_cleaned.txt for route_id keys in ROUTE_TERMINAL_STOPS.")
        return

    stop_names = load_stop_names(STOPS_FILE)

    trip_bounds: Dict[str, Tuple[int, str, str, int, str, str]] = {}
    for row in read_dict_rows(STOP_TIMES_CLEANED_FILE):
        trip_id = row.get('trip_id', '').strip()
        if trip_id not in trip_to_route:
            continue

        seq_raw = row.get('stop_sequence', '').strip()
        try:
            seq = int(seq_raw)
        except Exception:
            continue

        stop_id = row.get('stop_id', '').strip()
        dep_time = row.get('departure_time', '').strip()

        if trip_id not in trip_bounds:
            trip_bounds[trip_id] = (seq, stop_id, dep_time, seq, stop_id, dep_time)
            continue

        min_seq, min_sid, min_dep, max_seq, max_sid, max_dep = trip_bounds[trip_id]
        if seq < min_seq:
            min_seq, min_sid, min_dep = seq, stop_id, dep_time
        if seq > max_seq:
            max_seq, max_sid, max_dep = seq, stop_id, dep_time
        trip_bounds[trip_id] = (min_seq, min_sid, min_dep, max_seq, max_sid, max_dep)

    grouped_12: Dict[str, Dict[Tuple[str, str], Dict[str, object]]] = defaultdict(dict)
    grouped_34: Dict[str, Dict[Tuple[str, str], Dict[str, object]]] = defaultdict(dict)
    grouped_5: Dict[str, Dict[Tuple[str, str], Dict[str, object]]] = defaultdict(dict)
    grouped_0: Dict[str, Dict[Tuple[str, str], Dict[str, object]]] = defaultdict(dict)

    case_counts = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 0: 0}

    def add_group_item(
        bucket: Dict[str, Dict[Tuple[str, str], Dict[str, object]]],
        line_name: str,
        start_name: str,
        end_name: str,
        case_code: int,
        non_terminal_station_name: str,
        non_terminal_dep_time: str,
    ) -> None:
        pattern_key = (start_name, end_name)
        line_groups = bucket.setdefault(line_name, {})
        if pattern_key not in line_groups:
            line_groups[pattern_key] = {
                "count": 0,
                "cases": defaultdict(int),
                "times": defaultdict(list),
            }

        item = line_groups[pattern_key]
        item["count"] = int(item["count"]) + 1
        item["cases"][case_code] += 1
        if non_terminal_station_name:
            item["times"][non_terminal_station_name].append((non_terminal_dep_time or "")[:5])

    for trip_id in sorted(trip_bounds):
        route_id = trip_to_route[trip_id]
        line_name = route_short_name.get(route_id, route_id)

        expected_first, expected_last = ROUTE_TERMINAL_STOPS[route_id]
        expected_set = {expected_first, expected_last}

        _, min_sid, min_dep, _, max_sid, max_dep = trip_bounds[trip_id]

        start_name = stop_names.get(min_sid, "(no name)")
        end_name = stop_names.get(max_sid, "(no name)")

        if min_sid == expected_first and max_sid == expected_last:
            case_code = 1
        elif min_sid == expected_last and max_sid == expected_first:
            case_code = 2
        elif min_sid == expected_first and max_sid != expected_last:
            case_code = 3
        elif min_sid != expected_first and max_sid == expected_last:
            case_code = 4
        elif (min_sid not in expected_set) and (max_sid not in expected_set):
            case_code = 5
        else:
            case_code = 0

        case_counts[case_code] += 1

        if case_code in (1, 2):
            add_group_item(grouped_12, line_name, start_name, end_name, case_code, "", "")
            continue

        if case_code == 3:
            non_terminal_name = end_name if max_sid not in expected_set else ""
            add_group_item(grouped_34, line_name, start_name, end_name, case_code, non_terminal_name, max_dep)
            continue

        if case_code == 4:
            non_terminal_name = start_name if min_sid not in expected_set else ""
            add_group_item(grouped_34, line_name, start_name, end_name, case_code, non_terminal_name, min_dep)
            continue

        if case_code == 5:
            add_group_item(grouped_5, line_name, start_name, end_name, case_code, start_name, min_dep)
            add_group_item(grouped_5, line_name, start_name, end_name, case_code, end_name, max_dep)
            continue

        if case_code == 0:
            if min_sid not in expected_set:
                add_group_item(grouped_0, line_name, start_name, end_name, case_code, start_name, min_dep)
            if max_sid not in expected_set:
                add_group_item(grouped_0, line_name, start_name, end_name, case_code, end_name, max_dep)

    print("Case totals:")
    print(f"- Case 1 (exact): {case_counts[1]}")
    print(f"- Case 2 (reversed terminals): {case_counts[2]}")
    print(f"- Case 3 (only max mismatch): {case_counts[3]}")
    print(f"- Case 4 (only min mismatch): {case_counts[4]}")
    print(f"- Case 5 (none matches): {case_counts[5]}")
    print(f"- Other mixed cases: {case_counts[0]}")

    def print_bucket(
        title: str,
        bucket: Dict[str, Dict[Tuple[str, str], Dict[str, object]]],
        show_times: bool,
    ) -> None:
        total = sum(
            int(info["count"])
            for line_groups in bucket.values()
            for info in line_groups.values()
        )
        print(f"\n{title}: {total} trips")

        if total == 0:
            print("- none")
            return

        for line_name in sorted(bucket.keys()):
            line_groups = bucket[line_name]
            line_total = sum(int(v["count"]) for v in line_groups.values())
            print(f"Line {line_name}: {line_total}")

            sorted_patterns = sorted(
                line_groups.items(),
                key=lambda kv: (-int(kv[1]["count"]), kv[0][0], kv[0][1]),
            )

            for (start_name, end_name), info in sorted_patterns:
                count = int(info["count"])
                case_mix = ", ".join(
                    f"case {k}: {v}" for k, v in sorted(info["cases"].items())
                )
                print(f"- {count} began in {start_name} and ended in {end_name} ({case_mix})")

                if not show_times:
                    continue

                times_by_station = info["times"]
                for station_name in sorted(times_by_station.keys()):
                    times = sorted(t for t in times_by_station[station_name] if t)
                    unique_times = []
                    seen = set()
                    for t in times:
                        if t not in seen:
                            seen.add(t)
                            unique_times.append(t)

                    preview_limit = 12
                    shown = unique_times[:preview_limit]
                    suffix = "..." if len(unique_times) > preview_limit else ""
                    joined = ", ".join(shown) if shown else "(none)"
                    print(f"  departure times from non-terminal station {station_name}: {joined}{suffix}")

    print_bucket("Bucket (1,2)", grouped_12, show_times=False)
    print_bucket("Bucket (3,4)", grouped_34, show_times=True)
    print_bucket("Bucket 5", grouped_5, show_times=True)
    print_bucket("Other mixed cases", grouped_0, show_times=True)

main()